# Part VI: Proving the Lag-Absorption Hypothesis

## The finding this notebook tests

Part V measured whether SHAP recovers the item-level drivers the generator actually
used. The result split sharply:

| Driver | LightGBM | XGBoost |
|---|---|---|
| Price elasticity | rho = +0.632 (p = 0.0002) | rho = +0.638 (p = 0.0001) |
| Temperature | rho = +0.344 (p = 0.063) | rho = +0.300 (p = 0.107) |

Price recovery is strong. Temperature recovery is not significant for either model,
even though the generator gave every item an explicit `temp_effect` and both models
had temperature features available.

## Hypothesis

**H1 (lag absorption).** Temperature recovery is weak because rolling-mean features
already encode the weather response. `sales_mean_28d` carries the effect of the last
four weeks of temperature, so the model can predict accurately while attributing
almost nothing to the temperature columns themselves. SHAP stays faithful to the
model; the model has routed a real causal driver through the lags.

**H0.** Temperature recovery is weak for some other reason - the effect is simply too
small to detect - and removing lag features will not improve it.

## Pre-registered predictions

If H1 holds, then removing lag and rolling features should:

1. **Increase** temperature recovery rho substantially, ideally to significance.
2. **Decrease** forecast accuracy, since the lags are genuinely predictive.
3. Leave price recovery roughly unchanged, because price is not encoded in the lags
   in the same way - promotions are intermittent bursts rather than a slow-moving
   average.

Prediction 3 matters most. It is the falsification test: if removing lags improves
*everything*, the result is a generic regularisation effect rather than absorption.
Only a selective improvement supports H1.

## Setup

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import gc
import json
import os
import warnings

import lightgbm as lgbm
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import shap
import xgboost as xgb
from scipy import stats
from sklearn.metrics import mean_absolute_error, mean_squared_error

warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8-whitegrid")

SEED = 2025
rng = np.random.default_rng(SEED)

DATA_DIR = "../data"
FIGURES_DIR = "../figures"
os.makedirs(FIGURES_DIR, exist_ok=True)

N_EXPLAIN = 2000
N_BACKGROUND = 100
N_BOOTSTRAP = 2000

## Data and the held-out ground truth

In [ ]:
df = pd.read_parquet(os.path.join(DATA_DIR, "feature_engineered_data_69_features.parquet"))

DROP = ["date", "is_test", "store_item", "promo_id", "store_name", "item_name"]
TARGET = "sales"
all_features = [c for c in df.columns if c not in DROP + [TARGET]]

X_all = df[all_features].copy()
for c in all_features:
    if str(X_all[c].dtype) == "category":
        X_all[c] = X_all[c].cat.codes.astype("int32")
X_all = X_all.astype("float32")

train_mask = (~df["is_test"]).values
print(f"{len(df):,} rows, {len(all_features)} features")

In [ ]:
# The generator's item parameters. Never seen by any model - used only for
# validation, exactly as ground-truth player positions were in the reference
# clustering study.
item_truth = pd.DataFrame([
    (1, 0.000, -0.55, 7), (2, -0.004, -0.70, 7), (3, 0.000, -0.45, 7),
    (4, -0.008, -0.60, 7), (5, -0.006, -0.80, 7), (6, 0.000, -0.50, 7),
    (7, 0.002, -0.40, 1), (8, 0.000, -0.95, 7), (9, 0.010, -0.90, 1),
    (10, 0.045, -1.35, 1), (11, -0.006, -0.75, 7), (12, -0.025, -1.00, 7),
    (13, 0.035, -1.45, 1), (14, 0.020, -1.15, 1), (15, 0.042, -1.10, 1),
    (16, -0.024, -1.20, 7), (17, -0.020, -0.85, 7), (18, 0.050, -1.30, 1),
    (19, 0.015, -1.50, 1), (20, -0.005, -1.25, 7), (21, -0.018, -1.40, 7),
    (22, 0.000, -1.15, 7), (23, 0.004, -1.10, 1), (24, 0.000, -0.90, 7),
    (25, 0.000, -1.05, 7), (26, 0.000, -0.75, 7), (27, 0.000, -0.80, 7),
    (28, 0.060, -1.10, 1), (29, 0.045, -0.95, 1), (30, -0.045, -0.70, 7),
], columns=["item_id", "temp_effect", "elasticity", "peak_month"])

item_truth["abs_temp_effect"] = item_truth["temp_effect"].abs()
item_truth["abs_elasticity"] = item_truth["elasticity"].abs()
display(item_truth.head())

## The two feature arms

The only difference is the presence of lag, rolling and EWMA features. Everything
else - calendar, weather, price, promotion, store and item identifiers - is held
identical, so any change in recovery is attributable to the lags alone.

In [ ]:
LAG_PREFIXES = ("sales_lag_", "sales_mean_", "sales_min_", "sales_max_",
                "sales_std_", "sales_ewma_")
LAG_EXTRA = ["store_mean_7d", "item_mean_7d"]

lag_features = [c for c in all_features
                if c.startswith(LAG_PREFIXES) or c in LAG_EXTRA]
nolag_features = [c for c in all_features if c not in lag_features]

ARMS = {"with_lags": all_features, "no_lags": nolag_features}

print(f"with_lags : {len(all_features)} features")
print(f"no_lags   : {len(nolag_features)} features ({len(lag_features)} removed)")
print(f"\nremoved: {lag_features}")

## Feature groups mapped to known drivers

SHAP is per-feature; the ground truth is per-driver. Scoring recovery requires
aggregating SHAP mass into driver buckets, declared once here so the mapping is
explicit rather than buried in a loop.

In [ ]:
TEMP_FEATURES = [c for c in ["temp_anomaly", "temperature", "heat_excess",
                             "cold_excess", "temp_norm"] if c in all_features]
PRICE_FEATURES = [c for c in ["log_price_ratio", "discount_pct", "price",
                              "base_price", "is_deep_discount"] if c in all_features]
SEASON_FEATURES = [c for c in ["month_sin", "month_cos", "day_of_year", "month",
                               "week_of_year"] if c in all_features]

print("temperature :", TEMP_FEATURES)
print("price       :", PRICE_FEATURES)
print("seasonality :", SEASON_FEATURES)

## Run both arms

In [ ]:
def make_models(seed=SEED):
    return {
        "LightGBM": lgbm.LGBMRegressor(
            n_estimators=400, num_leaves=63, learning_rate=0.05,
            min_child_samples=40, subsample=0.85, subsample_freq=1,
            colsample_bytree=0.85, random_state=seed, n_jobs=-1, verbose=-1),
        "XGBoost": xgb.XGBRegressor(
            n_estimators=400, learning_rate=0.05, max_depth=8,
            min_child_weight=10, subsample=0.85, colsample_bytree=0.85,
            tree_method="hist", enable_categorical=False,
            random_state=seed, n_jobs=-1, verbosity=0),
    }


def group_mass(shap_values, cols, group, item_ids):
    """Mean |SHAP| over a feature group, per item."""
    idx = [cols.index(c) for c in group if c in cols]
    if not idx:
        return pd.Series(dtype=float)
    mass = np.abs(shap_values[:, idx]).sum(axis=1)
    return pd.Series(mass).groupby(item_ids).mean()


def run_arm(arm_name, cols):
    """Fit both models on one feature arm, score accuracy and driver recovery."""
    Xtr = X_all.loc[train_mask, cols]
    ytr = df.loc[train_mask, TARGET]
    Xte = X_all.loc[~train_mask, cols]
    yte = df.loc[~train_mask, TARGET]

    ei = np.sort(rng.choice(len(Xte), min(N_EXPLAIN, len(Xte)), replace=False))
    Xe = Xte.iloc[ei]
    item_ids = df.loc[~train_mask].iloc[ei]["item_id"].values
    bg = Xtr.iloc[rng.choice(len(Xtr), N_BACKGROUND, replace=False)]

    acc_rows, per_item = [], {}
    for mname, model in make_models().items():
        model.fit(Xtr, ytr)
        pred = model.predict(Xte)
        acc_rows.append({
            "arm": arm_name, "model": mname,
            "MAE": mean_absolute_error(yte, pred),
            "RMSE": np.sqrt(mean_squared_error(yte, pred)),
        })

        ex = shap.TreeExplainer(
            model, data=shap.maskers.Independent(bg, max_samples=len(bg)),
            feature_perturbation="interventional")
        sv = np.asarray(ex.shap_values(Xe, check_additivity=False))

        for driver, group in [("temperature", TEMP_FEATURES),
                              ("price", PRICE_FEATURES),
                              ("seasonality", SEASON_FEATURES)]:
            per_item[(mname, driver)] = group_mass(sv, cols, group, item_ids)

        del sv, model
        gc.collect()
        print(f"  {arm_name} / {mname} done")

    return pd.DataFrame(acc_rows), per_item


results_acc, results_shap = [], {}
for arm, cols in ARMS.items():
    print(f"\nrunning arm: {arm}")
    acc, per_item = run_arm(arm, cols)
    results_acc.append(acc)
    for k, v in per_item.items():
        results_shap[(arm, *k)] = v

accuracy = pd.concat(results_acc, ignore_index=True)
display(accuracy.round(4))

## Prediction 2 — the accuracy cost of removing lags

In [ ]:
pivot = accuracy.pivot(index="model", columns="arm", values="MAE")
pivot["cost_pct"] = 100 * (pivot["no_lags"] - pivot["with_lags"]) / pivot["with_lags"]
display(pivot.round(4))

print("\nPrediction 2 says removing lags should HURT accuracy.")
print("Confirmed:", bool((pivot["cost_pct"] > 0).all()))

## Predictions 1 and 3 — driver recovery

In [ ]:
TRUTH_COL = {"temperature": "abs_temp_effect", "price": "abs_elasticity"}

rows = []
for (arm, model, driver), series in results_shap.items():
    if driver not in TRUTH_COL:
        continue
    d = series.reset_index()
    d.columns = ["item_id", "shap_mass"]
    d = d.merge(item_truth, on="item_id")
    r = stats.spearmanr(d["shap_mass"], d[TRUTH_COL[driver]])
    rows.append({"arm": arm, "model": model, "driver": driver,
                 "spearman_rho": r.statistic, "p_value": r.pvalue,
                 "n_items": len(d)})

recovery = pd.DataFrame(rows)
table = recovery.pivot_table(index=["driver", "model"], columns="arm",
                             values="spearman_rho")
table["delta"] = table["no_lags"] - table["with_lags"]
display(table.round(4))
display(recovery.round(4))

### Is the change in rho larger than sampling noise?

Two Spearman correlations computed on the same 30 items are not independent, so a
naive test would be wrong. The bootstrap resamples **items** with replacement and
recomputes both rho values on the same resample, giving a confidence interval on
their difference directly.

In [ ]:
def bootstrap_delta(driver, model, n_boot=N_BOOTSTRAP, seed=SEED):
    tcol = TRUTH_COL[driver]
    a = results_shap[("with_lags", model, driver)]
    b = results_shap[("no_lags", model, driver)]
    d = (pd.DataFrame({"with_lags": a, "no_lags": b})
         .reset_index(names="item_id").merge(item_truth, on="item_id").dropna())

    boot_rng = np.random.default_rng(seed)
    n = len(d)
    deltas = np.empty(n_boot)
    for i in range(n_boot):
        idx = boot_rng.integers(0, n, n)
        s = d.iloc[idx]
        if s[tcol].nunique() < 3:
            deltas[i] = np.nan
            continue
        r_with = stats.spearmanr(s["with_lags"], s[tcol]).statistic
        r_no = stats.spearmanr(s["no_lags"], s[tcol]).statistic
        deltas[i] = r_no - r_with
    return deltas[np.isfinite(deltas)]


boot_rows = []
for driver in ["temperature", "price"]:
    for model in ["LightGBM", "XGBoost"]:
        deltas = bootstrap_delta(driver, model)
        lo, hi = np.percentile(deltas, [2.5, 97.5])
        boot_rows.append({
            "driver": driver, "model": model,
            "mean_delta": deltas.mean(), "ci_low": lo, "ci_high": hi,
            "p_delta_gt_0": (deltas > 0).mean(),
            "significant": bool(lo > 0 or hi < 0),
        })

boot = pd.DataFrame(boot_rows)
display(boot.round(4))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharey=True)
for ax, driver in zip(axes, ["temperature", "price"]):
    sub = table.loc[driver]
    x = np.arange(len(sub))
    ax.bar(x - 0.2, sub["with_lags"], 0.4, label="with lags", color="steelblue")
    ax.bar(x + 0.2, sub["no_lags"], 0.4, label="no lags", color="darkorange")
    ax.set_xticks(x)
    ax.set_xticklabels(sub.index)
    ax.axhline(0, color="black", linewidth=0.8)
    ax.set_title(f"{driver} recovery")
    ax.set_ylabel("Spearman rho vs true driver")
axes[0].legend()
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, "lag_absorption_recovery.png"), dpi=120)
plt.show()

## The accuracy / faithfulness trade-off

The central plot of the study. If lag features buy accuracy at the cost of
attribution faithfulness, the two arms sit at opposite corners, and the line between
them is the trade-off a practitioner actually faces.

In [ ]:
plot_rows = []
for arm in ARMS:
    for model in ["LightGBM", "XGBoost"]:
        mae = accuracy.query("arm == @arm and model == @model")["MAE"].iloc[0]
        rho = table.loc[("temperature", model), arm]
        plot_rows.append({"arm": arm, "model": model, "MAE": mae, "temp_rho": rho})
tradeoff = pd.DataFrame(plot_rows)

plt.figure(figsize=(8, 5.5))
for model, mk in [("LightGBM", "o"), ("XGBoost", "^")]:
    s = tradeoff[tradeoff["model"] == model]
    plt.plot(s["MAE"], s["temp_rho"], marker=mk, markersize=11,
             linewidth=1.5, label=model)
    for _, r in s.iterrows():
        plt.annotate(r["arm"], (r["MAE"], r["temp_rho"]),
                     textcoords="offset points", xytext=(8, 5), fontsize=9)

plt.xlabel("Test MAE (lower is better)")
plt.ylabel("Temperature recovery rho (higher is better)")
plt.title("Accuracy against attribution faithfulness")
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, "accuracy_faithfulness_tradeoff.png"), dpi=120)
plt.show()
display(tradeoff.round(4))

## Verdict

In [ ]:
temp_deltas = boot[boot["driver"] == "temperature"]
price_deltas = boot[boot["driver"] == "price"]

p1 = bool((temp_deltas["mean_delta"] > 0).all() and temp_deltas["significant"].any())
p2 = bool((pivot["cost_pct"] > 0).all())
p3 = bool(price_deltas["mean_delta"].abs().max() < temp_deltas["mean_delta"].abs().min())

print("PRE-REGISTERED PREDICTIONS\n")
print(f"1. Removing lags raises temperature recovery      : {p1}")
print(f"2. Removing lags costs forecast accuracy          : {p2}")
print(f"3. Price recovery changes less than temperature   : {p3}")
print()
if p1 and p2 and p3:
    print("VERDICT: H1 supported. Lag features absorb the temperature signal.")
    print("The models predict well while attributing the weather response to the")
    print("rolling means, and SHAP faithfully reports a model that has routed a")
    print("real causal driver through its lags.")
elif p1 and p2 and not p3:
    print("VERDICT: partial. Recovery improves but not selectively, so the effect")
    print("is more likely generic regularisation than targeted absorption.")
elif not p1:
    print("VERDICT: H1 not supported. Weak temperature recovery has another cause;")
    print("the most likely candidate is that the effect is too small relative to")
    print("noise to be detected across only 30 items.")
else:
    print("VERDICT: mixed. Read the bootstrap intervals before concluding.")

In [ ]:
out = {
    "accuracy": accuracy.to_dict("records"),
    "recovery": recovery.to_dict("records"),
    "bootstrap": boot.to_dict("records"),
    "predictions": {"p1_recovery_up": p1, "p2_accuracy_cost": p2,
                    "p3_selective": p3},
    "n_lag_features_removed": len(lag_features),
    "n_explain": N_EXPLAIN,
    "n_bootstrap": N_BOOTSTRAP,
}
with open(os.path.join(DATA_DIR, "lag_absorption_results.json"), "w") as f:
    json.dump(out, f, indent=2, default=float)

accuracy.to_csv(os.path.join(DATA_DIR, "lag_absorption_accuracy.csv"), index=False)
recovery.to_csv(os.path.join(DATA_DIR, "lag_absorption_recovery.csv"), index=False)
print("saved results")

## How to report this

**State the predictions before the results.** The design above is pre-registered in
the first markdown cell: the predictions are written down before any number is
computed. Say so in the paper. It is what separates this from a post-hoc story
fitted to whatever the data happened to show.

**Prediction 3 is the one that matters.** If removing lags improved every driver's
recovery equally, the explanation would be generic - fewer features, less
attribution dilution. Only a *selective* improvement in temperature, with price
roughly unchanged, supports absorption specifically.

**Report the trade-off, not a winner.** Neither arm is correct. The with-lags model
forecasts better; the no-lags model explains better. That tension is the
contribution, and it generalises to every lag-featured forecasting pipeline in
production, not just this dataset.

**Limitations.** Thirty items is a small sample for a Spearman correlation, which is
why the bootstrap reports an interval rather than a point estimate. The effect sizes
are set by the generator, so external validity requires repeating the protocol on
M5 or Favorita. And SHAP attributions remain associational: high attribution on
temperature is a statement about the model's function, not about causation.